# Chat Completions API Basics

The Chat Completions API is a powerful interface for generating text based on conversational prompts. It allows you to send a series of messages and receive a coherent response from the model. This guide will cover the basics of how to use the Chat Completions API, including how to structure your messages, interpret the response, and handle common limitations.

In [ ]:
import os
import gradio as gr
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

### Client Setup

The `OpenAI` client picks up `OPENAI_API_KEY` from the environment automatically if we do not provide it explicitly. To use a different provider, set `base_url` and `api_key` as needed.

```python
from openai import OpenAI
client = OpenAI()  # Uses OPENAI_API_KEY from environment
```

In [ ]:
# use OpenRouter key but OpenAI-compatible endpoint to demonstrate multi-provider setup
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
MODEL = "gpt-4o-mini"

### The Message List

Every request is a list of dicts with `role` and `content`.

| Role | Purpose |
|------|---------|
| `system` | Sets behavior and persona. Processed before any user input. |
| `user` | The human's question or instruction. |
| `assistant` | A previous model reply — used to reconstruct conversation history. |

In [ ]:
messages = [
    {"role": "system", "content": "Summarize the following content clearly and concisely."},
    {"role": "user",   "content": "Large language models are trained on vast corpora of text..."}
]

### Minimal API Call

`chat.completions.create()` sends the messages list and returns a `ChatCompletion` object.  
The generated text is at `choices[0].message.content`.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=messages
)

print(f"finish_reason     : {response.choices[0].finish_reason}")
print(f"prompt tokens     : {response.usage.prompt_tokens}")
print(f"completion tokens : {response.usage.completion_tokens}")
print(f"total tokens      : {response.usage.total_tokens}")
print()
print(response.choices[0].message.content)

### Response Structure

```
ChatCompletion
├── choices[0]
│   ├── finish_reason   "stop" | "length" | "tool_calls"
│   └── message
│       ├── role        "assistant"
│       └── content     ← the generated text
└── usage
    ├── prompt_tokens
    ├── completion_tokens
    └── total_tokens
```

| `finish_reason` | Meaning |
|----------------|---------|
| `stop` | Model finished naturally |
| `length` | Hit token limit — response may be cut off |
| `tool_calls` | Model wants to execute a function |

### Summarizer Function

Accepts a URL or plain text. Fetches and parses the page if a URL is given.  
`client` and `model` are parameters so the same function works across all providers.

In [ ]:
def summarize(source: str, client, model: str) -> str:
    """
    Accept a URL or plain text. Return an LLM-generated summary.
    Input is capped at 4000 characters to stay within a safe token budget.
    """
    if source.startswith("http"):
        html = requests.get(source, timeout=10).text
        text = BeautifulSoup(html, "html.parser").get_text(separator="\n", strip=True)
    else:
        text = source

    msgs = [
        {"role": "system", "content": "Summarize the following content clearly and concisely."},
        {"role": "user",   "content": text[:4000]}
    ]

    response = client.chat.completions.create(model=model, messages=msgs)
    return response.choices[0].message.content

### Summarizer

`gr.Interface` wraps the function in a web UI.  
Paste text or enter a URL — the summary renders as Markdown.

In [ ]:
def summarize_ui(source: str) -> str:
    return summarize(source, client, MODEL)


demo = gr.Interface(
    fn=summarize_ui,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Paste text or enter a URL...",
        label="Input"
    ),
    outputs=gr.Markdown(label="Summary"),
    title="Summarizer",
    description="Enter a URL or plain text. Returns a concise LLM-generated summary.",
    examples=[
        ["https://en.wikipedia.org/wiki/Large_language_model"],
        ["The transformer architecture replaced RNNs with self-attention, enabling parallel training."],
    ]
)

demo.launch()

### Multi-Provider Setup

The `OpenAI` client works with any OpenAI-compatible endpoint.  
Change `base_url`, `api_key`, and `model` — everything else stays the same.

| Provider | `base_url` | Env var | Approx cost (per 1M tokens) |
|----------|-----------|----------|---------|
| OpenAI | *(default)* | `OPENAI_API_KEY` | ~$0.15 in / $0.60 out (gpt-4o-mini) |
| Anthropic | `https://api.anthropic.com/v1` | `ANTHROPIC_API_KEY` | ~$0.80 in / $4.00 out (haiku) |
| Google Gemini | `https://generativelanguage.googleapis.com/v1beta/openai` | `GOOGLE_API_KEY` | ~$0.10 in / $0.40 out (flash) |
| Ollama (local) | `http://localhost:11434/v1` | any string | free — runs on your machine |

In [ ]:
PROVIDERS = {
    "openrouter": {
        "client": OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.getenv("OPENROUTER_API_KEY"),
        ),
        "model": "gpt-4o-mini",
    },
    "openai": {
        "client": OpenAI(),
        "model":  "gpt-4o-mini",
    },
    "claude": {
        "client": OpenAI(
            base_url="https://api.anthropic.com/v1",
            api_key=os.getenv("ANTHROPIC_API_KEY"),
        ),
        "model": "claude-3-5-haiku-20241022",
    },
    "gemini": {
        "client": OpenAI(
            base_url="https://generativelanguage.googleapis.com/v1beta/openai",
            api_key=os.getenv("GOOGLE_API_KEY"),
        ),
        "model": "gemini-2.0-flash",
    },
    "ollama": {
        "client": OpenAI(
            base_url="http://localhost:11434/v1",
            api_key="ollama",
        ),
        "model": "llama3.2",
    },
}

### Multi-Provider Comparison

`gr.Blocks` lays out a checkbox group + one Markdown output per provider.  
Type a prompt, select providers, click **Run** — responses appear side by side.

Ollama requires `ollama pull llama3.2` and a running local server.

In [ ]:
provider_names = list(PROVIDERS.keys())


def compare_providers(prompt: str, selected: list) -> list:
    """Run the same prompt across selected providers. Returns one reply per provider."""
    results = []
    for name in provider_names:
        if name not in selected:
            results.append("")
            continue
        cfg = PROVIDERS[name]
        try:
            results.append(summarize(prompt, cfg["client"], cfg["model"]))
        except Exception as e:
            results.append(f"**ERROR:** {e}")
    return results


with gr.Blocks(title="Multi-Provider Comparison") as comparison:
    gr.Markdown("## Multi-Provider Comparison")
    gr.Markdown("Enter a prompt, select providers, click **Run**. Responses appear side by side.")

    prompt_box = gr.Textbox(
        lines=3,
        placeholder="What is a large language model? Answer in one sentence.",
        label="Prompt"
    )
    provider_select = gr.CheckboxGroup(
        choices=provider_names,
        value=["openai"],
        label="Providers"
    )
    run_btn = gr.Button("Run", variant="primary")

    with gr.Row():
        outputs = [gr.Markdown(label=name) for name in provider_names]

    run_btn.click(
        fn=compare_providers,
        inputs=[prompt_box, provider_select],
        outputs=outputs
    )

comparison.launch()

### Limitations

| Limitation | Impact |
|-----------|--------|
| Token budget | `text[:4000]` is a rough guard, use `tiktoken` for precision |
| No memory | Each call is stateless; no conversation history in `summarize()` |
| Static content | `requests` + BeautifulSoup misses JavaScript-rendered pages |
| Provider drift | OpenAI-compatible endpoints differ slightly; some parameters may be ignored |
| Cost | Every call consumes tokens; long inputs and large models cost more |